# Machine Learning 1 — Práctica 7: Support Vector Machines (SVM)
### Facultad de Ingeniería — Universidad Nacional de Asunción (FIUNA)
**Docente Teoría:** Diego Stalder | **Docente Práctica:** Carlos Benítez

---

## Descripción y Objetivos Pedagógicos
En este cuaderno exploraremos en profundidad las **Máquinas de Vectores de Soporte (Support Vector Machines - SVM)**, uno de los algoritmos de aprendizaje supervisado más potentes, robustos y matemáticamente elegantes de la inteligencia artificial.

Basado rigurosamente en el **Capítulo 6 de las notas de CS229 (Andrew Ng - Stanford University)**, abordaremos los siguientes aspectos centrales:
1. **Márgenes Funcional y Geométrico:** Definición formal y comprensión intuitiva de la distancia al hiperplano separador $w^T x + b = 0$.
2. **Clasificador de Margen Óptimo (Hard-Margin SVM):** Formulación del problema de optimización primal y su resolución mediante Mínimos Cuadrados / Programación Cuadrática.
3. **Dualidad de Lagrange y Condiciones KKT:** Construcción del Lagrangiano primal, derivación del problema dual $W(\alpha)$ y demostración empírica de la complementariedad dual (los Vectores de Soporte son los únicos con $\alpha_i > 0$).
4. **El Truco del Kernel (Kernel Trick):** Mapeo de características a espacios de dimensión superior $\phi(x)$, Teorema de Mercer y comparación de Kernels (Lineal, Polinomial, RBF/Gausiano y Sigmoide).
5. **Soft-Margin SVM y Regularización ($C$):** Introducción de variables de holgura $\xi_i$ y análisis del trade-off entre maximizar el margen y tolerar violaciones de margen.
6. **Algoritmo SMO (Sequential Minimal Optimization):** Implementación desde cero en Python del solver de John Platt para actualizar analíticamente pares de multiplicadores $(\alpha_i, \alpha_j)$.
7. **Práctica en Scikit-Learn y Tuning con `GridSearchCV`:** Aplicación de `SVC`, `SVR`, `StandardScaler` en `Pipeline` y búsqueda automatizada de hiperparámetros.

---

In [ ]:
# Configuración del entorno de trabajo e importación de librerías científicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, mean_squared_error

# Configuración estética de gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 100

print('Entorno de Machine Learning 1 (FIUNA) cargado exitosamente para la Práctica de SVM.')

---
## 1. Generación de Datos y Concepto de Margen Separador

De acuerdo con la notación de Andrew Ng (CS229 Sec. 6.2), las etiquetas binarias se denotan como $y^{(i)} \in \{-1, +1\}$.
Un hiperplano separador se parametriza mediante un vector de pesos $w \in \mathbb{R}^d$ y un sesgo $b \in \mathbb{R}$:

$$h_{w,b}(x) = g(w^T x + b), \quad \text{donde } g(z) = \begin{cases} +1 & \text{si } z \ge 0 \\ -1 & \text{si } z < 0 \end{cases}$$

* **Margen Funcional:** $\hat{\gamma}^{(i)} = y^{(i)}(w^T x^{(i)} + b)$. Mide la confianza de la predicción.
* **Margen Geométrico:** $\gamma^{(i)} = y^{(i)} \left( \frac{w^T x^{(i)} + b}{\|w\|} \right) = \frac{\hat{\gamma}^{(i)}}{\|w\|}$. Mide la distancia física euclidiana al hiperplano.

In [ ]:
# 1. Generación de un conjunto de datos linealmente separable en 2D
X_sep, y_sep = make_blobs(n_samples=40, centers=2, random_state=42, cluster_std=0.85)
# Convertir etiquetas de {0, 1} a {-1, +1}
y_sep = np.where(y_sep == 0, -1, 1)

# Función de utilidad para calcular márgenes funcional y geométrico
def calcular_margenes(X, y, w, b):
    norm_w = np.linalg.norm(w)
    margen_funcional = y * (X @ w + b)
    margen_geometrico = margen_funcional / norm_w
    return margen_funcional, margen_geometrico

# Ejemplo hipotético con pesos w e intercepto b
w_demo = np.array([0.8, -1.2])
b_demo = 0.5

mf, mg = calcular_margenes(X_sep, y_sep, w_demo, b_demo)
print(f'Margen Funcional Mínimo del dataset: {np.min(mf):.4f}')
print(f'Margen Geométrico Mínimo del dataset (Distancia euclidiana mínima): {np.min(mg):.4f}')

---
## 2. Clasificador de Margen Óptimo (Hard-Margin SVM Primal)

Para encontrar la frontera con la máxima separación posible respecto a las dos clases, resolvemos el problema de **Programación Cuadrática Canónico** (CS229 Sec. 6.4):

$$\min_{w, b} \frac{1}{2} \|w\|^2 \quad \text{sujeto a} \quad y^{(i)}(w^T x^{(i)} + b) \ge 1, \quad i=1,\dots,m$$

A continuación, resolveremos este problema primal numéricamente mediante `scipy.optimize.minimize` para constatar cómo la optimización matemática ubica los hiperplanos exactos.

In [ ]:
# Solución numérica directa del problema Primal Hard-Margin
def funcion_objetivo_primal(params):
    w = params[:-1]
    return 0.5 * np.dot(w, w)

def restriccion_margen(params, X, y):
    w = params[:-1]
    b = params[-1]
    return y * (X @ w + b) - 1.0  # debe ser >= 0

n_samples, n_features = X_sep.shape
init_params = np.zeros(n_features + 1)

constraints = {'type': 'ineq', 'fun': restriccion_margen, 'args': (X_sep, y_sep)}
res_primal = minimize(funcion_objetivo_primal, init_params, constraints=constraints, method='SLSQP')

w_opt = res_primal.x[:-1]
b_opt = res_primal.x[-1]

print('=== Solución Primal SVM (Hard Margin) ===')
print(f'w* óptimo: {w_opt}')
print(f'b* óptimo: {b_opt:.4f}')
print(f'Ancho del Margen (2 / ||w||): {2.0 / np.linalg.norm(w_opt):.4f}')

# Graficar hiperplano óptimo y márgenes
fig, ax = plt.subplots(figsize=(8, 6))
x0_min, x0_max = X_sep[:, 0].min() - 1, X_sep[:, 0].max() + 1
x1_min, x1_max = X_sep[:, 1].min() - 1, X_sep[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x0_min, x0_max, 200), np.linspace(x1_min, x1_max, 200))
Z = (np.c_[xx.ravel(), yy.ravel()] @ w_opt + b_opt).reshape(xx.shape)

ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['#EF4444', '#1E293B', '#2563EB'],
           linestyles=['--', '-', '--'], linewidths=[2, 2.8, 2])
ax.scatter(X_sep[y_sep==1, 0], X_sep[y_sep==1, 1], c='#2563EB', s=70, label='Clase +1')
ax.scatter(X_sep[y_sep==-1, 0], X_sep[y_sep==-1, 1], c='#EF4444', s=70, marker='s', label='Clase -1')

# Identificar vectores de soporte (puntos donde y*(w^T x + b) ≈ 1)
margenes_opt = y_sep * (X_sep @ w_opt + b_opt)
sv_idx = np.where(np.isclose(margenes_opt, 1.0, atol=1e-3))[0]
ax.scatter(X_sep[sv_idx, 0], X_sep[sv_idx, 1], s=200, facecolors='none', edgecolors='#F59E0B', linewidths=2.5, label='Vectores de Soporte')

ax.set_title('Clasificador de Margen Óptimo (Solución Primal Exacta)', fontweight='bold')
ax.legend()
plt.show()

---
## 3. Formulación Dual y Dualidad de Lagrange (Condiciones KKT)

Aplicando multiplicadores de Lagrange $\alpha_i \ge 0$ (CS229 Sec. 6.5-6.6), el problema dual se formula como:

$$\max_{\alpha} W(\alpha) = \sum_{i=1}^m \alpha_i - \frac{1}{2} \sum_{i=1}^m \sum_{j=1}^m y^{(i)} y^{(j)} \alpha_i \alpha_j \langle x^{(i)}, x^{(j)} \rangle$$
$$\text{sujeto a} \quad \alpha_i \ge 0, \quad i=1,\dots,m \quad \text{y} \quad \sum_{i=1}^m \alpha_i y^{(i)} = 0$$

### Consecuencia Crítica de la Complementariedad Dual (KKT):
$$\alpha_i \left[ y^{(i)}(w^T x^{(i)} + b) - 1 \right] = 0$$
* Si el punto $x^{(i)}$ está fuera del margen ($y^{(i)}(w^T x^{(i)}+b) > 1$), **necesariamente $\alpha_i = 0$**.
* Solo los puntos exactamente sobre el margen tienen $\alpha_i > 0$. ¡Son los **Vectores de Soporte**!

In [ ]:
# Solución Dual de SVM Hard Margin
def funcion_objetivo_dual(alpha, X, y):
    # Minimizar -W(alpha) para equivaler a maximizar W(alpha)
    Gram = (X @ X.T)
    target_matrix = np.outer(y, y) * Gram
    return 0.5 * np.sum(np.outer(alpha, alpha) * target_matrix) - np.sum(alpha)

# Restricción sum(alpha_i * y_i) = 0
def restriccion_dual(alpha, y):
    return np.dot(alpha, y)

bounds = [(0, None) for _ in range(n_samples)]
constraints_dual = {'type': 'eq', 'fun': restriccion_dual, 'args': (y_sep,)}
init_alpha = np.zeros(n_samples)

res_dual = minimize(funcion_objetivo_dual, init_alpha, args=(X_sep, y_sep), method='SLSQP', bounds=bounds, constraints=constraints_dual)
alpha_opt = res_dual.x
# Filtrar pequeños errores numéricos
alpha_opt[alpha_opt < 1e-5] = 0.0

# Reconstrucción de w* y b* a partir de los multiplicadores duales alpha
w_dual = np.sum((alpha_opt * y_sep)[:, np.newaxis] * X_sep, axis=0)
sv_mask = alpha_opt > 1e-4
b_dual = np.mean(y_sep[sv_mask] - X_sep[sv_mask] @ w_dual)

print('=== Solución Dual SVM (Condiciones KKT) ===')
print(f'Número de Vectores de Soporte (alpha_i > 0): {np.sum(sv_mask)} de {n_samples}')
print(f'w* recuperado desde el Dual: {w_dual}')
print(f'b* recuperado desde el Dual: {b_dual:.4f}')
print(f'Diferencia ||w_primal - w_dual||: {np.linalg.norm(w_opt - w_dual):.2e}')

---
## 4. El Truco del Kernel (Kernel Trick) & Funciones Kernel

Para resolver problemas no separables linealmente (CS229 Sec. 6.7), reemplazamos el producto interno $\langle x^{(i)}, x^{(j)} \rangle$ por una **Función Kernel** $K(x^{(i)}, x^{(j)}) = \langle \phi(x^{(i)}), \phi(x^{(j)}) \rangle$:

* **Kernel Lineal:** $K(x, z) = x^T z$
* **Kernel Polinomial:** $K(x, z) = (x^T z + c)^d$
* **Kernel RBF / Gausiano:** $K(x, z) = \exp\left( -\gamma \|x - z\|^2 \right)$
* **Kernel Sigmoide:** $K(x, z) = \tanh(\alpha x^T z + c)$

Por el **Teorema de Mercer**, cualquier función cuya **Matriz Gram** $K_{ij} = K(x^{(i)}, x^{(j)})$ sea simétrica y semi-definida positiva ($K \succeq 0$) es un Kernel válido.

In [ ]:
# Implementación manual de Funciones Kernel y Matriz Gram
class CustomKernels:
    @staticmethod
    def linear(X1, X2):
        return X1 @ X2.T
    
    @staticmethod
    def polynomial(X1, X2, degree=3, coef0=1.0):
        return (X1 @ X2.T + coef0) ** degree
    
    @staticmethod
    def rbf(X1, X2, gamma=1.0):
        # Calculo eficiente de distancia euclidiana al cuadrado ||x - z||^2
        dists = np.sum(X1**2, axis=1)[:, np.newaxis] + np.sum(X2**2, axis=1) - 2 * (X1 @ X2.T)
        return np.exp(-gamma * dists)

# Demostración en un dataset no lineal (Concentric Circles)
X_circ, y_circ = make_circles(n_samples=180, factor=0.3, noise=0.08, random_state=42)
y_circ_svm = np.where(y_circ == 0, -1, 1)

K_lin = CustomKernels.linear(X_circ, X_circ)
K_rbf = CustomKernels.rbf(X_circ, X_circ, gamma=2.0)

print('Verificación de la Matriz Gram RBF:')
print(f'Forma de la matriz K: {K_rbf.shape}')
print(f'¿Es simétrica?: {np.allclose(K_rbf, K_rbf.T)}')
eigenvalues = np.linalg.eigvalsh(K_rbf)
print(f'¿Es semi-definida positiva (mín autovalor >= 0)?: {np.min(eigenvalues) >= -1e-8}')

---
## 5. Soft-Margin SVM y Regularización ($C$)

En presencia de ruido o datos solapados (CS229 Sec. 6.8), introducimos **variables de holgura** $\xi_i \ge 0$:

$$\min_{w, b, \xi} \frac{1}{2} \|w\|^2 + C \sum_{i=1}^m \xi_i \quad \text{s.t.} \quad y^{(i)}(w^T x^{(i)} + b) \ge 1 - \xi_i, \quad \xi_i \ge 0$$

En el problema dual, esto impone una **restricción de caja (box constraint)** a los multiplicadores de Lagrange:

$$0 \le \alpha_i \le C, \quad i=1,\dots,m$$

* **$C$ Grande ($C \to \infty$):** Penaliza severamente las violaciones. Produce un margen estrecho y aumenta el riesgo de **sobreajuste (Overfitting)**.
* **$C$ Pequeño ($C \to 0$):** Permite mayores violaciones de margen. Produce un margen ancho y robusto a ruido, con riesgo de **subajuste (Underfitting)**.

In [ ]:
# Experimento: Impacto del hiperparámetro C en la frontera de decisión
X_moon, y_moon = make_moons(n_samples=150, noise=0.25, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
c_list = [0.01, 1.0, 100.0]

for ax, C_val in zip(axes, c_list):
    clf = SVC(kernel='rbf', C=C_val, gamma=1.0)
    clf.fit(X_moon, y_moon)
    
    x0_m, x0_M = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    x1_m, x1_M = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x0_m, x0_M, 200), np.linspace(x1_m, x1_M, 200))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.3)
    ax.contour(xx, yy, Z, levels=[0], colors=['#1E293B'], linewidths=[2.5])
    ax.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap='bwr', edgecolors='k', s=45)
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=120, facecolors='none', edgecolors='#F59E0B', linewidths=1.5)
    
    ax.set_title(f'Soft Margin: C = {C_val}\n({len(clf.support_vectors_)} Vectores de Soporte)', fontweight='bold')

plt.suptitle('Efecto del Parámetro de Regularización C (RBF Kernel)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Algoritmo SMO (Sequential Minimal Optimization) Simplificado

El algoritmo **SMO de John Platt** (CS229 Sec. 6.9) resuelve el problema dual eficientemente mediante **Subida de Coordenadas (Coordinate Ascent)** actualizando **pares de multiplicadores** $(\alpha_1, \alpha_2)$ analíticamente.

Dado que $\alpha_1 y^{(1)} + \alpha_2 y^{(2)} = \zeta$, los nuevos valores deben situarse dentro de la caja $[0, C] \times [0, C]$ entre las cotas $L$ y $H$:

* **Si $y^{(1)} \neq y^{(2)}$:** $L = \max(0, \alpha_2 - \alpha_1), \quad H = \min(C, C + \alpha_2 - \alpha_1)$
* **Si $y^{(1)} = y^{(2)}$:** $L = \max(0, \alpha_2 + \alpha_1 - C), \quad H = \min(C, \alpha_2 + \alpha_1)$

A continuación se presenta un solver SMO simplificado escrito en Python puro.

In [ ]:
# Solver SMO Simplificado (John Platt - CS229 Section 6.9)
class SimpleSMOSolver:
    def __init__(self, C=1.0, tol=1e-3, max_passes=5, kernel='linear', gamma=1.0):
        self.C = C
        self.tol = tol
        self.max_passes = max_passes
        self.kernel_type = kernel
        self.gamma = gamma
        
    def _kernel(self, X1, X2):
        if self.kernel_type == 'linear':
            return X1 @ X2.T
        elif self.kernel_type == 'rbf':
            dists = np.sum(X1**2, axis=1)[:, np.newaxis] + np.sum(X2**2, axis=1) - 2 * (X1 @ X2.T)
            return np.exp(-self.gamma * dists)
    
    def fit(self, X, y):
        m, n = X.shape
        self.X = X
        self.y = y.astype(float)
        self.alpha = np.zeros(m)
        self.b = 0.0
        K = self._kernel(X, X)
        
        passes = 0
        while passes < self.max_passes:
            num_changed_alphas = 0
            for i in range(m):
                E_i = np.sum(self.alpha * self.y * K[:, i]) + self.b - self.y[i]
                
                # Verificación de violación de condiciones KKT
                if (self.y[i] * E_i < -self.tol and self.alpha[i] < self.C) or \
                   (self.y[i] * E_i > self.tol and self.alpha[i] > 0):
                    
                    # Seleccionar j != i aleatoriamente
                    j = i
                    while j == i:
                        j = np.random.randint(0, m)
                    
                    E_j = np.sum(self.alpha * self.y * K[:, j]) + self.b - self.y[j]
                    alpha_i_old, alpha_j_old = self.alpha[i], self.alpha[j]
                    
                    # Calcular cotas L y H
                    if self.y[i] != self.y[j]:
                        L = max(0.0, self.alpha[j] - self.alpha[i])
                        H = min(self.C, self.C + self.alpha[j] - self.alpha[i])
                    else:
                        L = max(0.0, self.alpha[i] + self.alpha[j] - self.C)
                        H = min(self.C, self.alpha[i] + self.alpha[j])
                    
                    if L == H:
                        continue
                    
                    eta = 2.0 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0:
                        continue
                    
                    # Actualizar alpha_j sin recortar y luego recortar a [L, H]
                    self.alpha[j] -= (self.y[j] * (E_i - E_j)) / eta
                    self.alpha[j] = np.clip(self.alpha[j], L, H)
                    
                    if abs(self.alpha[j] - alpha_j_old) < 1e-5:
                        continue
                    
                    # Actualizar alpha_i
                    self.alpha[i] += self.y[i] * self.y[j] * (alpha_j_old - self.alpha[j])
                    
                    # Actualizar b
                    b1 = self.b - E_i - self.y[i]*(self.alpha[i] - alpha_i_old)*K[i,i] - self.y[j]*(self.alpha[j] - alpha_j_old)*K[i,j]
                    b2 = self.b - E_j - self.y[i]*(self.alpha[i] - alpha_i_old)*K[i,j] - self.y[j]*(self.alpha[j] - alpha_j_old)*K[j,j]
                    
                    if 0 < self.alpha[i] < self.C:
                        self.b = b1
                    elif 0 < self.alpha[j] < self.C:
                        self.b = b2
                    else:
                        self.b = (b1 + b2) / 2.0
                    
                    num_changed_alphas += 1
            
            passes = (passes + 1) if num_changed_alphas == 0 else 0
            
        return self

    def predict(self, X_test):
        K_test = self._kernel(X_test, self.X)
        vals = np.sum((self.alpha * self.y) * K_test, axis=1) + self.b
        return np.sign(vals)
# Probar nuestro Solver SMO casero vs Sklearn SVC
smo = SimpleSMOSolver(C=1.0, kernel='rbf', gamma=1.0).fit(X_moon, y_moon*2 - 1)
preds_smo = np.where(smo.predict(X_moon) == -1, 0, 1)
acc_smo = accuracy_score(y_moon, preds_smo)
clf_sk = SVC(C=1.0, kernel='rbf', gamma=1.0).fit(X_moon, y_moon)
acc_sk = accuracy_score(y_moon, clf_sk.predict(X_moon))
print(f'Exactitud en Entrenamiento (Nuestro Solver SMO): {acc_smo * 100:.2f}%')
print(f'Exactitud en Entrenamiento (Scikit-Learn SVC):     {acc_sk * 100:.2f}%')

---
## 7. Optimización de Hiperparámetros con `GridSearchCV` y `Pipeline`

Debido a que SVM depende de distancias en el espacio de características, **el escalado con `StandardScaler` es obligatorio**.
Para evitar la **Fuga de Datos (Data Leakage)** durante la Validación Cruzada, acoplaremos las etapas en un `sklearn.pipeline.Pipeline` y ejecutaremos una Búsqueda en Malla (`GridSearchCV`).

In [ ]:
# División Train/Test y construcción de Pipeline
X_train, X_test, y_train, y_test = train_test_split(X_moon, y_moon, test_size=0.3, random_state=42, stratify=y_moon)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])

param_grid = {
    'svc__C': [0.1, 1.0, 10.0, 100.0],
    'svc__gamma': [0.01, 0.1, 1.0, 10.0],
    'svc__kernel': ['rbf', 'poly']
}

grid_search = GridSearchCV(pipe, param_grid, cv=StratifiedKFold(n_splits=5), scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('=== Mejores Hiperparámetros Encontrados ===')
print(grid_search.best_params_)
print(f'Mejor Exactitud en Validación Cruzada (CV): {grid_search.best_score_*100:.2f}%')

# Evaluación en el conjunto de prueba (Holdout Test)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print('\n=== Reporte de Clasificación en Test ===')
print(classification_report(y_test, y_pred))

# Mapa de calor de GridSearchCV para RBF
results_df = pd.DataFrame(grid_search.cv_results_)
rbf_results = results_df[results_df['param_svc__kernel'] == 'rbf']
scores_matrix = rbf_results.pivot(index='param_svc__C', columns='param_svc__gamma', values='mean_test_score')

plt.figure(figsize=(7, 5))
sns.heatmap(scores_matrix, annot=True, fmt='.3f', cmap='viridis', cbar_kws={'label': 'CV Accuracy'})
plt.title('Mapa de Calor de Validación Cruzada (GridSearchCV C vs Gamma)', fontweight='bold')
plt.xlabel('Gamma (Ancho de Banda RBF)')
plt.ylabel('C (Regularización Soft Margin)')
plt.show()

---
## 8. Support Vector Regression (SVR) y Clasificación Multiclase

### Support Vector Regression (SVR)
En lugar de buscar un hiperplano separador, SVR busca una función $f(x) = w^T x + b$ que encierre las muestras dentro de una banda de margen de ancho $\varepsilon$ (tubo insensible a errores $\varepsilon$-insensitive loss).

### Clasificación Multiclase ($K > 2$)
* **One-vs-One (OvO):** Entrena $\frac{K(K-1)}{2}$ clasificadores binarios y decide por votación.
* **One-vs-Rest (OvR):** Entrena $K$ clasificadores (cada clase contra todas las demás).

In [ ]:
# 1. Ejemplo de Support Vector Regression (SVR) en una función no lineal sin(x)
np.random.seed(42)
X_svr = np.sort(np.random.uniform(-3, 3, (80, 1)), axis=0)
y_svr = np.sin(X_svr).ravel() + np.random.normal(0, 0.1, 80)

svr_rbf = SVR(kernel='rbf', C=10.0, epsilon=0.1, gamma=0.5)
svr_rbf.fit(X_svr, y_svr)

X_plot = np.linspace(-3.5, 3.5, 200).reshape(-1, 1)
y_plot = svr_rbf.predict(X_plot)

plt.figure(figsize=(9, 5))
plt.scatter(X_svr, y_svr, color='#2563EB', label='Muestras con Ruido', s=40)
plt.plot(X_plot, y_plot, color='#DC2626', linewidth=2.5, label='Predicción SVR (RBF)')
plt.plot(X_plot, y_plot + 0.1, color='#DC2626', linestyle='--', alpha=0.6, label='Tubo epsilon-insensitive (+/- 0.1)')
plt.plot(X_plot, y_plot - 0.1, color='#DC2626', linestyle='--', alpha=0.6)
plt.scatter(X_svr[svr_rbf.support_], y_svr[svr_rbf.support_], s=120, facecolors='none', edgecolors='#F59E0B', linewidths=2, label='Vectores de Soporte SVR')
plt.title('Support Vector Regression (SVR) con Tubo Insensible a Errores', fontweight='bold')
plt.legend()
plt.show()

# 2. Ejemplo Multiclase en Dataset Iris (OvO vs OvR)
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)

svc_ovo = SVC(decision_function_shape='ovo', kernel='rbf', C=1.0).fit(X_tr_i, y_tr_i)
svc_ovr = SVC(decision_function_shape='ovr', kernel='rbf', C=1.0).fit(X_tr_i, y_tr_i)

print(f'Exactitud Multiclase Iris (One-vs-One - OvO):  {svc_ovo.score(X_te_i, y_te_i)*100:.2f}%')
print(f'Exactitud Multiclase Iris (One-vs-Rest - OvR): {svc_ovr.score(X_te_i, y_te_i)*100:.2f}%')

---
## 9. Ejercicios Prácticos Guiados

### Ejercicio 1: Kernel Laplaciano Personalizado
El Kernel Laplaciano se define como $K(x, z) = \exp\left( -\gamma \|x - z\|_1 \right)$ (usando la norma $L_1$ o distancia Manhattan).
Implemente una función Kernel Laplaciano personalizada y pásela como argumento `kernel=funcion_kernel` a `sklearn.svm.SVC` para evaluar su desempeño en el conjunto `X_moon`.

### Ejercicio 2: Manejo de Desbalance de Clases con `class_weight`
Generar un conjunto de datos altamente desbalanceado ($90\%$ clase 0, $10\%$ clase 1). Comparar la frontera de decisión y el recall de la clase minoritaria usando `SVC(C=1.0)` vs `SVC(C=1.0, class_weight='balanced')`.

### Ejercicio 3: Comparación de Tiempos de Cómputo vs Tamaño del Dataset ($m$)
Evaluar el tiempo de entrenamiento de `LinearSVC` (basado en LIBLINEAR, $O(m)$) vs `SVC(kernel='linear')` (basado en LIBSVM, $O(m^2)$ a $O(m^3)$) haciendo crecer el número de muestras $m \in [1000, 5000, 10000, 20000]$.

In [ ]:
# Soluciones a los Ejercicios Guiados

# --- Ejercicio 1: Kernel Laplaciano Custom ---
def laplacian_kernel(X1, X2, gamma=1.0):
    # Calculo de distancia L1 en 2D
    dists = np.sum(np.abs(X1[:, np.newaxis, :] - X2[np.newaxis, :, :]), axis=2)
    return np.exp(-gamma * dists)

clf_laplacian = SVC(kernel=laplacian_kernel, C=1.0)
clf_laplacian.fit(X_train, y_train)
acc_lap = clf_laplacian.score(X_test, y_test)
print(f'=== Ejercicio 1: Exactitud con Kernel Laplaciano Custom: {acc_lap*100:.2f}% ===')

# --- Ejercicio 2: Desbalance de Clases ---
X_unbal, y_unbal = make_blobs(n_samples=[450, 50], centers=2, random_state=42, cluster_std=1.5)
clf_std = SVC(kernel='rbf', C=1.0).fit(X_unbal, y_unbal)
clf_bal = SVC(kernel='rbf', C=1.0, class_weight='balanced').fit(X_unbal, y_unbal)

print('\n=== Ejercicio 2: Sensibilidad al Desbalance de Clases ===')
print(f'Vectores de Soporte Estándar: {len(clf_std.support_vectors_)}')
print(f'Vectores de Soporte Balanc.:  {len(clf_bal.support_vectors_)}')
print('Recall Clase Minoritaria (Estándar):', classification_report(y_unbal, clf_std.predict(X_unbal), output_dict=True)['1']['recall'])
print('Recall Clase Minoritaria (Balanced):', classification_report(y_unbal, clf_bal.predict(X_unbal), output_dict=True)['1']['recall'])

---
## 10. Conclusiones y Resumen Pedagógico

1. **Principio del Margen Máximo:** SVM no solo busca separar las clases, sino maximizar el margen geométrico $\gamma = \frac{2}{\|w\|}$, garantizando la mayor capacidad de generalización teórica (menor riesgo empírico).
2. **Dualidad y Esparsidad:** Gracias a las condiciones KKT, el modelo final depende **exclusivamente de los Vectores de Soporte** (puntos donde $\alpha_i > 0$). Todos los demás puntos pueden eliminarse sin alterar la frontera.
3. **Poder No Lineal vía Kernels:** El Kernel Trick computa el producto interno en un espacio transformado $\mathcal{H}$ sin necesidad de construir explícitamente el vector $\phi(x)$, posibilitando fronteras complejas con bajo costo computacional.
4. **Soft Margin y Regularización:** El hiperparámetro $C$ controla el compromiso entre la varianza y el sesgo. Un $C$ elevado fuerza un margen estricto, mientras que un $C$ modesto proporciona inmunidad ante valores atípicos (outliers).
5. **Buenas Prácticas de Ingeniería:**
   * **Escalado:** Siempre aplicar `StandardScaler` sobre las variables antes de entrenar SVM.
   * **Data Leakage:** Envolver el escalador y la SVM dentro de un `Pipeline` antes de hacer `GridSearchCV`.
   * **Eficiencia:** Para datasets grandes ($m > 50,000$), utilizar `LinearSVC` o SGDClassifier en lugar de `SVC` con RBF.